In [ ]:
# Data loader and preprocessing block. 
# It reads this aligned data from S3, splits it using a rolling 6-month temporal window, 
# and outputs the exact tensors needed by the Bidirectional LSTM.

import io
import boto3
import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler

BUCKET_NAME = "us-recession-model-bucket"
S3_PREFIX = "data/"
FILES = {
    "UNRATE": "fred_unrate.csv",
    "FEDFUNDS": "fred_fedfunds.csv",
    "GDP": "fred_gdp.csv",
    "INFLATION": "fred_inflation.csv",
    "RECESSION": "fred_recession.csv"
}

def load_and_prepare_training_tensors(sequence_length=6):
    s3_client = boto3.client("s3")
    data_frames = {}
    
    # 1. Gather files straight from S3 memory streams
    for feature_name, filename in FILES.items():
        file_key = f"{S3_PREFIX}{filename}"
        response = s3_client.get_object(Bucket=BUCKET_NAME, Key=file_key)
        df = pd.read_csv(io.StringIO(response["Body"].read().decode('utf-8')))
        df['date'] = pd.to_datetime(df['date'])
        df.set_index('date', inplace=True)
        if 'value' in df.columns:
            df.rename(columns={'value': feature_name}, inplace=True)
        data_frames[feature_name] = df

    # 2. Join into a single structured master dataframe
    feature_keys = list(FILES.keys())
    df_master = data_frames[feature_keys[0]]
    for feature_name in feature_keys[1:]:
        df_master = df_master.join(data_frames[feature_name], how="inner")
        
    # 3. Separate the features (X) from the target label vector (y)
    X_raw = df_master[['UNRATE', 'FEDFUNDS', 'GDP', 'INFLATION']].values
    y_raw = df_master['RECESSION'].values
    
    # 4. Standardize features (Z-Score scaling prevents GDP from dominating the gradients)
    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X_raw)
    
    # 5. Slide a rolling window across the dataset to build 3D temporal sequences
    X_sequences = []
    y_labels = []
    
    for i in range(len(X_scaled) - sequence_length):
        # Extract 6 consecutive months of scaled features
        X_sequences.append(X_scaled[i : i + sequence_length])
        # Target label maps directly to the final month of that 6-month sequence window
        y_labels.append(y_raw[i + sequence_length])
        
    X = np.array(X_sequences, dtype=np.float32)
    y = np.array(y_labels, dtype=np.float32)
    
    print("=" * 70)
    print(" TRAINING TENSORS SUCCESSFULLY PREPARED")
    print(f"   • Input Feature Matrix Shape (X): {X.shape} -> (Samples, Timesteps, Features)")
    print(f"   • Target Ground Truth Label Shape (y): {y.shape}")
    print("=" * 70)
    
    return X, y, scaler

# Execute the tensor loading routine
X, y, trained_scaler = load_and_prepare_training_tensors(sequence_length=6)

🚀 TRAINING TENSORS SUCCESSFULLY PREPARED
   • Input Feature Matrix Shape (X): (542, 6, 4) -> (Samples, Timesteps, Features)
   • Target Ground Truth Label Shape (y): (542,)


In [ ]:
# End-to-End Model Training Script.

# It integrates prepared 3D tensors directly into the Bidirectional LSTM architecture,
# fits the parameters, and saves the final result as a clean production SavedModel.


import os
import tensorflow as tf
from tensorflow.keras import backend as K
from tensorflow.keras.layers import Input, Bidirectional, LSTM, Dropout, Dense
from tensorflow.keras.models import Model

# 1. Clear background graph memory metrics
K.clear_session()

# 2. Rebuild the network layers using the clean Functional API layout
inputs = Input(shape=(6, 4), name="input_lstm")
x = Bidirectional(LSTM(64, return_sequences=True))(inputs)
x = Dropout(0.2)(x)
x = Bidirectional(LSTM(32, return_sequences=False))(x)
x = Dropout(0.2)(x)
x = Dense(16, activation='relu')(x)
outputs = Dense(1, activation='sigmoid')(x)

# Bind model boundaries
model = Model(inputs=inputs, outputs=outputs)
model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])

print("Retraining model layers...")
model.fit(X, y, epochs=15, batch_size=16, validation_split=0.1, verbose=1)

# 3. THE UTILITY FIX: Use Keras 3 native export() to write a perfect SavedModel folder
export_path = "./export/7"
if os.path.exists(export_path):
    import shutil
    shutil.rmtree(export_path)

# This compiles perfectly for TensorFlow Serving with completely clean graph signatures
model.export(export_path)
print(f"Model natively exported and written to: {export_path}")

2026-10-27 06:40:56.025278: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX512F, in other operations, rebuild TensorFlow with the appropriate compiler flags.


🚀 Retraining model layers...
Epoch 1/15
31/31 ━━━━━━━━━━━━━━━━━━━━ 7s 39ms/step - accuracy: 0.8932 - loss: 0.4643 - val_accuracy: 0.8545 - val_loss: 0.4076
Epoch 2/15
31/31 ━━━━━━━━━━━━━━━━━━━━ 0s 15ms/step - accuracy: 0.9220 - loss: 0.2374 - val_accuracy: 0.8182 - val_loss: 0.3352
Epoch 3/15
31/31 ━━━━━━━━━━━━━━━━━━━━ 0s 15ms/step - accuracy: 0.9261 - loss: 0.2031 - val_accuracy: 0.7818 - val_loss: 0.3288
Epoch 4/15
31/31 ━━━━━━━━━━━━━━━━━━━━ 1s 17ms/step - accuracy: 0.9363 - loss: 0.1867 - val_accuracy: 0.8545 - val_loss: 0.2744
Epoch 5/15
31/31 ━━━━━━━━━━━━━━━━━━━━ 1s 15ms/step - accuracy: 0.9322 - loss: 0.1856 - val_accuracy: 0.8909 - val_loss: 0.2278
Epoch 6/15
31/31 ━━━━━━━━━━━━━━━━━━━━ 1s 14ms/step - accuracy: 0.9446 - loss: 0.1723 - val_accuracy: 0.8909 - val_loss: 0.2027
Epoch 7/15
31/31 ━━━━━━━━━━━━━━━━━━━━ 0s 14ms/step - accuracy: 0.9425 - loss: 0.1799 - val_accuracy: 1.0000 - val_loss: 0.0981
Epoch 8/15
31/31 ━━━━━━━━━━━━━━━━━━━━ 0s 14ms/step - accuracy: 0.9446 - loss: 0.16

INFO:tensorflow:Assets written to: ./export/7/assets


Saved artifact at './export/7'. The following endpoints are available:

* Endpoint 'serve'
  args_0 (POSITIONAL_ONLY): TensorSpec(shape=(None, 6, 4), dtype=tf.float32, name='input_lstm')
Output Type:
  TensorSpec(shape=(None, 1), dtype=tf.float32, name=None)
Captures:
  140360513289488: TensorSpec(shape=(), dtype=tf.resource, name=None)
  140360513289680: TensorSpec(shape=(), dtype=tf.resource, name=None)
  140360513289296: TensorSpec(shape=(), dtype=tf.resource, name=None)
  140360513288720: TensorSpec(shape=(), dtype=tf.resource, name=None)
  140360513288912: TensorSpec(shape=(), dtype=tf.resource, name=None)
  140360513290064: TensorSpec(shape=(), dtype=tf.resource, name=None)
  140360511194896: TensorSpec(shape=(), dtype=tf.resource, name=None)
  140360511194512: TensorSpec(shape=(), dtype=tf.resource, name=None)
  140360511196624: TensorSpec(shape=(), dtype=tf.resource, name=None)
  140360511196816: TensorSpec(shape=(), dtype=tf.resource, name=None)
  140360511196048: TensorSpec(s